# Final — Pendulum with Discretized Q-Learning
### Continuous Control · State Discretization · Q-Table
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Pendulum-v1 — Q-Learning via Discretization

**Challenge:** Continuous state `[cos θ, sin θ, θ̇]` and continuous action `torque ∈ [-2,2]`.

**Solution:** Discretize into a finite grid then apply standard Q-Learning.

```
State bins:   10 × 10 × 10  →  1,000 discrete states
Action bins:  [-2, -1, 0, +1, +2]  →  5 discrete actions
Q-table:      shape [10, 10, 10, 5]
```

In [ ]:
import numpy as np, gymnasium as gym, matplotlib.pyplot as plt

NUM_A=5; BINS=[10,10,10]; A_SPACE=np.linspace(-2,2,NUM_A)
env=gym.make("Pendulum-v1"); Q=np.zeros(BINS+[NUM_A])
ALPHA=0.1; GAMMA=0.99; EPS=0.2; EPISODES=5_000

def disc(state):
    c,s,d=state
    edges=[np.linspace(-1,1,BINS[0]+1)[1:-1],np.linspace(-1,1,BINS[1]+1)[1:-1],np.linspace(-8,8,BINS[2]+1)[1:-1]]
    return tuple(np.digitize(v,e) for v,e in zip(state,edges))

rets=[]
print(f"Training Q-Learning on Pendulum ({EPISODES:,} episodes)...")
for ep in range(EPISODES):
    s,_=env.reset(); sd=disc(s); tot=0.
    for _ in range(200):
        ai=np.random.choice(NUM_A) if np.random.rand()<EPS else np.argmax(Q[sd])
        ns,r,done,_,_=env.step(np.array([A_SPACE[ai]])); nd=disc(ns)
        Q[sd][ai]+=ALPHA*(r+GAMMA*Q[nd][np.argmax(Q[nd])]-Q[sd][ai])
        sd=nd; tot+=r
        if done: break
    rets.append(tot)
    if (ep+1)%1000==0: print(f"  Ep {ep+1:>5} | avg={np.mean(rets[-1000:]):.1f}")
env.close()
w=500; plt.figure(figsize=(10,4))
plt.plot(np.convolve(rets,np.ones(w)/w,'valid'))
plt.axhline(-200,color='r',ls='--',label='~-200 threshold')
plt.xlabel("Episode"); plt.ylabel("Reward"); plt.title("Pendulum — Discretized Q-Learning")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Final avg: {np.mean(rets[-500:]):.1f}  (random≈-1200, upright=0)")
